In [ ]:
%pip install ultralytics

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

class HybridThermalSentinel:
    def __init__(self, model_path, min_aspect_ratio=0.8, max_human_pixel=235, temp_threshold=37.5):
        """
        :param model_path: Eğitilmiş YOLOv8 model yolu (.pt, .onnx veya .engine)
        :param min_aspect_ratio: İnsan silueti için minimum Yükseklik/Genişlik oranı (H/W)
        :param max_human_pixel: İnsan vücudunun üretebileceği maks piksel değeri (235 üstü araç motorudur)
        :param temp_threshold: Anomali kabul edilecek sıcaklık eşiği (°C)
        """
        self.model = YOLO(model_path)
        self.min_aspect_ratio = min_aspect_ratio
        self.max_human_pixel = max_human_pixel
        self.temp_threshold = temp_threshold

    def apply_clahe(self, gray_frame):
        """1. Kademeli çözüm: Koyu insanları ve düşük kontrastlı bölgeleri belirginleştirir."""
        clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
        return clahe.apply(gray_frame)
    
    def is_valid_human_geometry(self, bbox):
        """2. Kademli çözüm: En-Boy Oranı ile yatay araç/kaput kutularını eler."""
        x1, y1, x2, y2 = bbox
        w = x2 - x1
        h = y2 - y1
        if w <= 0 or h <= 0:
            return False
        aspect_ratio = h / float(w)
        # İnsanlar dikine uzanır (H > W), araç/kaput kutuları ise genelde yataydır
        return aspect_ratio >= self.min_aspect_ratio
    
    def is_within_biological_heat(self, roi):
        """3. Kademeli çözüm: Aşırı ısınmış araç motoru/egzoz gibi pikselleri eler."""
        if roi.size == 0:
            return False
        # Eğer kutu içindeki pikseller 235'in (aşırı parlak/metal sıcaklığı) üzerindeyse insana ait olamaz
        if np.max(roi) > self.max_human_pixel:
            return False
        return True
    
    def calculate_temperature(self, roi):
        """Koyu giysilerden etkilenmemek için %10 tepe sıcaklık hesabı."""
        top_10_percent = np.percentile(roi, 90)
        avg_hottest_pixels = np.mean(roi[roi >= top_10_percent])
        
        # Dönüştürme formülü (30°C taban + piksel katsayısı)
        estimated_temp = 30.0 + (avg_hottest_pixels * 0.04)
        is_anomaly = estimated_temp >= self.temp_threshold
        return round(float(estimated_temp), 1), is_anomaly

    def process_and_compare(self, image_path, save_output_path="comparison_result.jpg"):
        # 1. Görüntüyü gri tonlamalı olarak oku
        gray_frame = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        if gray_frame is None:
            raise FileNotFoundError(f"Görsel bulunamadı: {image_path}")

        # Karşılaştırma için ham / işlenmemiş kopyayı renkli yapalım
        raw_colored = cv2.cvtColor(gray_frame, cv2.COLOR_GRAY2BGR)

        # 2. Kademe 1: CLAHE Uygula (koyu giysili insanları keskinleştir)
        enhanced_gray = self.apply_clahe(gray_frame)
        
        # Görselleştirme için Inferno renk haritası uygula
        inferno_frame = cv2.applyColorMap(enhanced_gray, cv2.COLORMAP_INFERNO)

        # 3. YOLOv8 tespiti (Geliştirilmiş gri resim üzerinden)
        results = self.model(enhanced_gray, verbose=False)[0]

        rejected_boxes = []
        valid_boxes = []

        for box in results.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            bbox = [x1, y1, x2, y2]
            
            # Kutunun içindeki pikselleri kes (Region of Interest)
            roi = enhanced_gray[y1:y2, x1:x2]

            # Kademe 2: Geometri Kontrolü (araba/kaput Elenmesi)
            if not self.is_valid_human_geometry(bbox):
                rejected_boxes.append((bbox, "Elendi: En-Boy Oranı (Araç/Yatay)"))
                continue

            # Kademe 3: Termal tavan kontrolü (motor/egzoz elenmesi)
            if not self.is_within_biological_heat(roi):
                rejected_boxes.append((bbox, "Elendi: Aşırı Sıcaklık (Motor/Metal)"))
                continue

            # Tüm filtreleri geçen insan için sıcaklık hesabı
            temp, is_anomaly = self.calculate_temperature(roi)
            valid_boxes.append((bbox, temp, is_anomaly))


        #  Görsel çizimleri
        
        # A. Ham resim üzerine filtresiz/direkt YOLO sonuçlarını çiz (sol taraf)
        for box in results.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            cv2.rectangle(raw_colored, (x1, y1), (x2, y2), (255, 255, 255), 2)
            cv2.putText(raw_colored, "Ham YOLO", (x1, max(y1-5, 15)), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

        # B. Hibrit filtreli resim çizimi (sağ taraf)
        # Elenen kutuları Mor renkli çizelim
        for bbox, reason in rejected_boxes:
            x1, y1, x2, y2 = bbox
            cv2.rectangle(inferno_frame, (x1, y1), (x2, y2), (255, 0, 255), 2)
            cv2.putText(inferno_frame, "ELENDI", (x1, max(y1-5, 15)), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 0, 255), 1)

        # Onaylanan insan Kutularını çizelim (Normal: Yeşil, Anomali: Kırmızı)
        for bbox, temp, is_anomaly in valid_boxes:
            x1, y1, x2, y2 = bbox
            color = (0, 0, 255) if is_anomaly else (0, 255, 0)
            label = f"Insan: {temp}C"
            cv2.rectangle(inferno_frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(inferno_frame, label, (x1, max(y1-5, 15)), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

        # Yan yana karşılaştırma (befor/after)
        combined_view = np.hstack((raw_colored, inferno_frame))
        
        # Başlık ekleme
        cv2.putText(combined_view, "1. Ham model tespiti (Filtresiz)", (20, 30), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        cv2.putText(combined_view, "2. 3-Kademli hybrid sentinel (CLAHE + Filtering)", (gray_frame.shape[1] + 20, 30), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

        # Kaydet ve göster
        cv2.imwrite(save_output_path, combined_view)
        print(f"[BAŞARILI] Karşılaştırmalı görsel kaydedildi: {save_output_path}")
        
        return combined_view

In [ ]:
# Dosya yolları 
MODEL_PATH = "/content/yolov8_gold_best.pt"
IMAGE_PATH = "/content/050211.jpg"

print("İşlem başlatılıyor...") 

# 2. Sınıfı Başlatma
sentinel = HybridThermalSentinel(
    model_path=MODEL_PATH,
    min_aspect_ratio=0.8, # Yatay kutuları (araç/kaput) ele
    max_human_pixel=235,   # Araç motor parlamasını ele
    temp_threshold=37.5    # Anomali sıcaklık eşiği (°C)
)

# 3. Test ve Karşılaştırma İşlemi
output_img = sentinel.process_and_compare(
    image_path=IMAGE_PATH,
    save_output_path="/content/050211_hybrid_result.jpg"
)

# 4. Sonucu doğrudan Colab hücresinin altında gösteriyoruz
plt.figure(figsize=(18, 9))
# OpenCV BGR okuduğu için Matplotlib RGB formatına çeviriyoruz
plt.imshow(cv2.cvtColor(output_img, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("Output: Ham model (sol) vs 3-kademeli hibrit sentinel (sağ)")
plt.show()